# quark quickstart

Embed quantum circuits, compare them, find duplicates in a folder, retrieve similar ones from a library. ~5 minute walkthrough.

In [ ]:
import torch
from qiskit import QuantumCircuit
from quark import CircuitEncoder, embed, equiv
from quark.pairs import random_circuit, rewrite_chain
import random

## 1. load a pretrained model

If you already trained one with `quark train`, point at the .pt. Otherwise this notebook just initializes random weights for illustration; numbers will be lower.

In [ ]:
enc = CircuitEncoder()
try:
    enc.load_state_dict(torch.load('../quark.pt', map_location='cpu', weights_only=True))
    print(f'loaded trained weights, {enc.n_params():,} params')
except FileNotFoundError:
    print(f'no weights found, using random init ({enc.n_params():,} params) — train first for real numbers')

## 2. embed two circuits and check similarity

These two circuits are equivalent up to a redundant pair of Hadamards. The trained model gives them a high similarity (around 0.9+), well above the ~0.55 typical of unrelated circuits.

In [ ]:
a = QuantumCircuit(3); a.h(0); a.cx(0, 1); a.cx(1, 2)
b = QuantumCircuit(3); b.h(0); b.h(0); b.h(0); b.cx(0, 1); b.cx(1, 2)

ea, eb = embed(enc, [a, b])
print(f'similarity (a, b): {(ea * eb).sum().item():.4f}')
print(f'verified equivalent: {equiv(a, b)}')

## 3. compare against a different (but valid) circuit

An unrelated 3-qubit circuit should land much further away.

In [ ]:
c = QuantumCircuit(3); c.x(0); c.cx(0, 2); c.h(1)
ec = embed(enc, [c])[0]
print(f'similarity (a, c): {(ea * ec).sum().item():.4f}')
print(f'verified equivalent: {equiv(a, c)}')

## 4. retrieve from a small library

Build 50 random circuits, hide 2 known-equivalents of a query in there, ask for the top 5 by similarity.

In [ ]:
query = random_circuit(3, 12, seed=0)
library = []
truth = []

rng = random.Random(99)
for j in range(2):
    sub = random.Random(j)
    eq_circ = rewrite_chain(query, k=4, rng=sub)
    truth.append(len(library))
    library.append(eq_circ)

for i in range(48):
    library.append(random_circuit(rng.randint(2, 4), rng.randint(8, 24), seed=99 + i))

perm = list(range(len(library)))
rng.shuffle(perm)
library = [library[i] for i in perm]
truth = [perm.index(t) for t in truth]

q_emb = embed(enc, [query])
lib_emb = embed(enc, library)
sims = (q_emb @ lib_emb.t()).squeeze(0)
top = sims.topk(5).indices.tolist()
print(f'top-5 retrieved: {top}')
print(f'truth equivalents at indices: {truth}')
print(f'hits: {len(set(top) & set(truth))} / {len(truth)}')

## 5. directory deduplication

Point at a folder of QASM files, find groups of equivalents.

In [ ]:
from quark.dedupe import cluster, fmt_report, _walk

paths = list(_walk('/tmp/qasm_test'))
if paths:
    res = cluster(paths, weights='../quark.pt', threshold=0.9)
    print(fmt_report(res))
else:
    print('no QASM files in /tmp/qasm_test — generate some first')

## next

- `scripts/heldout.py` — held-out-rewrite generalization test
- `scripts/qasmbench.py` — real corpus benchmark on QASMBench
- `quark train --help` — train your own model